In [ ]:
import time
import copy
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from collections import Counter
from PIL import Image
from tqdm import tqdm

*setup*

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

raw_dataset = r"C:\Users\Sina_Si1\Desktop\Project Code\Phase 2\PlantVillage"  # original dataset
balanced_dataset = r"C:\Users\Sina_Si1\Desktop\Project Code\Phase 2\PV_Balanced"  # after augmentation
split_output = r"C:\Users\Sina_Si1\Desktop\Project Code\Phase 2\PV_Split"  # train/val/test split

*Count Class Samples*

In [ ]:
def plot_class_distribution(dataset_path, title):
    class_counts = {cls: len(os.listdir(os.path.join(dataset_path, cls)))
                    for cls in os.listdir(dataset_path)}
    plt.figure(figsize=(15, 5))
    plt.bar(class_counts.keys(), class_counts.values())
    plt.xticks(rotation=90)
    plt.title(title)
    plt.show()
    return class_counts

print("Original class distribution:")
orig_counts = plot_class_distribution(raw_dataset, "Original dataset distribution")

*Augmentation*

In [ ]:
print("Balanced class distribution:")
balanced_counts = plot_class_distribution(balanced_dataset, "Balanced dataset distribution")

In [ ]:
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.Resize((224, 224))
])

os.makedirs(balanced_dataset, exist_ok=True)
max_count = max(orig_counts.values())

for cls, count in orig_counts.items():
    src_dir = os.path.join(raw_dataset, cls)
    dst_dir = os.path.join(balanced_dataset, cls)
    os.makedirs(dst_dir, exist_ok=True)

    # Copy original
    for img in os.listdir(src_dir):
        shutil.copy(os.path.join(src_dir, img), os.path.join(dst_dir, img))

    # Augment until balanced
    imgs = os.listdir(src_dir)
    idx = 0
    while len(os.listdir(dst_dir)) < max_count:
        img = Image.open(os.path.join(src_dir, imgs[idx % len(imgs)])).convert("RGB")
        img_aug = transform_aug(img)
        save_path = os.path.join(dst_dir, f"aug_{idx}.png")
        img_aug.save(save_path)
        idx += 1

print("Balanced class distribution:")
balanced_counts = plot_class_distribution(balanced_dataset, "Balanced dataset distribution")

*Split Dataset*

In [ ]:
def split_dataset(input_dir, output_dir, split_ratio=(0.6, 0.2, 0.2)):
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    classes = os.listdir(input_dir)
    for cls in classes:
        cls_dir = os.path.join(input_dir, cls)
        images = os.listdir(cls_dir)
        train, temp = train_test_split(images, test_size=1-split_ratio[0], random_state=42)
        val, test = train_test_split(temp, test_size=0.5, random_state=42)

        for split, files in zip(["train", "val", "test"], [train, val, test]):
            split_dir = os.path.join(output_dir, split, cls)
            os.makedirs(split_dir, exist_ok=True)
            for f in files:
                shutil.copy(os.path.join(cls_dir, f), os.path.join(split_dir, f))

split_dataset(balanced_dataset, split_output)

*Data Loaders*

In [ ]:
train_dir = os.path.join(split_output, "train")
val_dir = os.path.join(split_output, "val")
test_dir = os.path.join(split_output, "test")

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder(train_dir, transform=train_transforms)
val_data = datasets.ImageFolder(val_dir, transform=val_test_transforms)
test_data = datasets.ImageFolder(test_dir, transform=val_test_transforms)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print("Dataset ready with", len(train_data), "train,", len(val_data), "val,", len(test_data), "test samples.")

*Training Loop*

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=7):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    train_loss, val_loss = [], []
    train_acc, val_acc = [], []

    for epoch in range(epochs):
        print(f'Epoch {epoch+1}/{epochs}')
        print('-' * 20)

        for phase, loader in [('train', train_loader), ('val', val_loader)]:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            loop = tqdm(loader, desc=f"{phase} Epoch {epoch+1}/{epochs}", leave=False)

            for inputs, labels in loop:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

                loop.set_postfix(loss=loss.item())

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc = running_corrects.double() / len(loader.dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'train':
                train_loss.append(epoch_loss)
                train_acc.append(epoch_acc.item())
            else:
                val_loss.append(epoch_loss)
                val_acc.append(epoch_acc.item())

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model, {"train_loss": train_loss, "val_loss": val_loss,
                   "train_acc": train_acc, "val_acc": val_acc}

*Evaluation*

In [ ]:
def evaluate_model(model, test_loader, model_name):
    model.eval()
    preds, labels_list = [], []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            preds.extend(predicted.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    print(f"Classification Report for {model_name}:")
    print(classification_report(labels_list, preds))

    cm = confusion_matrix(labels_list, preds)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=False, cmap="Blues")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

    return {
        "report": classification_report(labels_list, preds, output_dict=True),
        "confusion_matrix": cm
    }

**Run Models**

In [ ]:
results = {}

In [ ]:
# ResNet50
print("Training ResNet50...")
model_resnet = models.resnet50(weights="IMAGENET1K_V1")
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, len(train_data.classes))
model_resnet = model_resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_resnet.parameters(), lr=0.001)

model_resnet, hist_resnet = train_model(model_resnet, train_loader, val_loader, criterion, optimizer, epochs=7)
results["ResNet50"] = evaluate_model(model_resnet, test_loader, "ResNet50")

In [ ]:
# EfficientNetB0
print("Training EfficientNetB0...")
model_eff = models.efficientnet_b0(weights="IMAGENET1K_V1")
model_eff.classifier[1] = nn.Linear(model_eff.classifier[1].in_features, len(train_data.classes))
model_eff = model_eff.to(device)

optimizer_eff = optim.Adam(model_eff.parameters(), lr=0.001)

model_eff, hist_eff = train_model(model_eff, train_loader, val_loader, criterion, optimizer_eff, epochs=7)
results["EfficientNetB0"] = evaluate_model(model_eff, test_loader, "EfficientNetB0")

In [ ]:
# MobileNetV2
print("Training MobileNetV2...")
model_mob = models.mobilenet_v2(weights="IMAGENET1K_V1")
model_mob.classifier[1] = nn.Linear(model_mob.classifier[1].in_features, len(train_data.classes))
model_mob = model_mob.to(device)

optimizer_mob = optim.Adam(model_mob.parameters(), lr=0.001)

model_mob, hist_mob = train_model(model_mob, train_loader, val_loader, criterion, optimizer_mob, epochs=7)
results["MobileNetV2"] = evaluate_model(model_mob, test_loader, "MobileNetV2")